# 08 - Adaptation variants (pre-registered Task 3)

LightGBM only, same 6 evaluation origins and the same breaks detected at each origin (data up to the origin only) as notebook 02. Definitions fixed in `docs/PREREGISTRATION.md`:

| variant | items with a break detected by the origin |
|---|---|
| (a) truncate | train only from the latest change week (existing "adaptive", min. 56 days) |
| (b) decay weights | weight 0.5^(days before the change week / 182) before it, 1 after |
| (c) level-shift feature | add `post_break` and `weeks_since_break` (capped at 104) |
| (d) blend | 0.5 × static + 0.5 × truncate |

Items without a break keep the static forecast. **Helps** = Holm-adjusted p < 0.05, median per-item MASE change < 0, and at least 0.0074 in size. Groups: all detected breaks, and the no-lookahead high-confidence subset.

In [1]:
import sys, time
sys.path.append('../src')
import numpy as np
import pandas as pd

from break_detection import to_weekly, weekly_ca_prices
from calibration import pelt_changepoints
from break_quality import assess_break_at_origin
from forecasting import build_lgb_panel, lgb_forecast
from metrics import paired_test, holm_adjust

BURN_IN, REF_LEN = 8, 52
PRACTICAL = 0.0074
PEN = float(pd.read_csv("../outputs/tables/break_calibration.csv").set_index('method').loc['PELT', 'value'])

fc = pd.read_parquet("../data/processed/backtest_forecasts.parquet")
lgb_fc = fc[fc['model'] == 'lightgbm'][['item_id', 'origin', 'date', 'variant', 'forecast', 'actual']]
err = pd.read_csv("../data/processed/backtest_errors.csv", parse_dates=['origin'])
scales = err[['item_id', 'origin', 'mase_scale']].drop_duplicates()
windows = pd.read_csv("../data/processed/breaks_by_origin.csv",
                      parse_dates=['origin', 'first_date', 'last_change_week', 'last_alarm_week', 'adaptive_start', 'placebo_start'])
ORIGINS = sorted(windows['origin'].unique())
EVAL = ORIGINS[1:]

long_df = pd.concat([pd.read_csv(f"../data/interim/{c}_candidates.csv", parse_dates=['date'])
                     for c in ['foods', 'hobbies', 'household']], ignore_index=True).sort_values(['item_id', 'date'])
category = long_df.groupby('item_id')['item_id'].first().str.split('_').str[0]
daily = {i: g[['date', 'sales']].reset_index(drop=True) for i, g in long_df.groupby('item_id')}
print(len(daily), "items;", "eval origins:", [o.date() for o in EVAL])

726 items; eval origins: [datetime.date(2015, 12, 6), datetime.date(2016, 1, 3), datetime.date(2016, 1, 31), datetime.date(2016, 2, 28), datetime.date(2016, 3, 27), datetime.date(2016, 4, 24)]


## No-lookahead high-confidence flag at each origin

In [2]:
rows = []
for r in windows[windows['origin'].isin(EVAL) & windows['has_break']].itertuples():
    d = daily[r.item_id]
    w = to_weekly(d[d['date'] <= r.origin])
    c = w.index.get_loc(r.last_change_week)
    a = assess_break_at_origin(w.to_numpy(), c, pelt_changepoints(w.to_numpy(), PEN, BURN_IN, REF_LEN))
    rows.append({'item_id': r.item_id, 'origin': r.origin, **a})
hc = pd.DataFrame(rows)
hc.to_csv("../outputs/tables/hc_by_origin.csv", index=False)
print(f"item-origins with a break: {len(hc)}; high-confidence: {int(hc['high_confidence'].sum())} "
      f"({hc['high_confidence'].mean():.1%}); items high-confidence at >=1 origin: {hc.loc[hc['high_confidence'], 'item_id'].nunique()}")
print(hc[['agreement', 'effect', 'persistence']].mean().round(3).to_string())

item-origins with a break: 2869; high-confidence: 1172 (40.9%); items high-confidence at >=1 origin: 254
agreement      0.648
effect         0.872
persistence    0.650


## Check: the refactored LightGBM reproduces the saved static forecasts

In [3]:
calendar = pd.read_csv("../data/raw/calendar.csv", parse_dates=['date'])
prices = pd.read_csv("../data/raw/sell_prices.csv", dtype={'store_id': 'category'})
price_wk = weekly_ca_prices(prices, calendar, long_df['item_id'].unique())
del prices
panel = build_lgb_panel(long_df, calendar, price_wk)

check, _ = lgb_forecast(panel, EVAL[-1])
saved = lgb_fc[(lgb_fc['origin'] == EVAL[-1]) & (lgb_fc['variant'] == 'static')]
m = check.merge(saved, on=['item_id', 'date'], suffixes=('_new', '_saved'))
print(f"rows compared: {len(m)}, max abs difference: {np.abs(m['forecast_new'] - m['forecast_saved']).max():.2e}")

rows compared: 20328, max abs difference: 0.00e+00


## Train variants (b) and (c) at each evaluation origin

In [4]:
parts = []
t0 = time.time()
for origin in EVAL:
    w = windows[(windows['origin'] == origin) & windows['has_break']]
    brk = dict(zip(w['item_id'], w['last_change_week']))
    for variant, kw in [('decay', {'decay_break_by_item': brk}), ('level_shift', {'shift_break_by_item': brk})]:
        out, _ = lgb_forecast(panel, origin, **kw)
        out['origin'], out['variant'] = origin, variant
        parts.append(out)
    print(f"{origin.date()} done ({time.time() - t0:.0f}s)")
new = pd.concat(parts, ignore_index=True)

ev = lgb_fc[lgb_fc['origin'].isin(EVAL)]
static = ev[ev['variant'] == 'static'][['item_id', 'origin', 'date', 'forecast', 'actual']]
trunc = ev[ev['variant'] == 'adaptive'][['item_id', 'origin', 'date', 'forecast']].rename(columns={'forecast': 'truncate'})
wide = static.rename(columns={'forecast': 'static'}).merge(trunc, on=['item_id', 'origin', 'date'])
for v in ['decay', 'level_shift']:
    wide = wide.merge(new[new['variant'] == v][['item_id', 'origin', 'date', 'forecast']].rename(columns={'forecast': v}),
                      on=['item_id', 'origin', 'date'])
wide['blend'] = 0.5 * wide['static'] + 0.5 * wide['truncate']
wide = wide.merge(windows[['item_id', 'origin', 'has_break']], on=['item_id', 'origin'])
for v in ['decay', 'level_shift']:              # items without a break keep the static forecast
    wide.loc[~wide['has_break'], v] = wide.loc[~wide['has_break'], 'static']
wide.to_parquet("../data/processed/variant_forecasts.parquet", index=False)
print(len(wide), "rows")

2015-12-06 done (15s)


2016-01-03 done (34s)


2016-01-31 done (50s)


2016-02-28 done (67s)


2016-03-27 done (84s)


2016-04-24 done (101s)


121968 rows


## Score and test

In [5]:
VARIANTS = ['truncate', 'decay', 'level_shift', 'blend']
mae = wide.groupby(['item_id', 'origin']).apply(
    lambda g: pd.Series({v: np.mean(np.abs(g['actual'] - g[v])) for v in ['static'] + VARIANTS}), include_groups=False).reset_index()
mase = mae.merge(scales, on=['item_id', 'origin'])
for v in ['static'] + VARIANTS:
    mase[v] = mase[v] / mase['mase_scale']
chk = mase.merge(err[(err['model'] == 'lightgbm') & (err['variant'] == 'static')][['item_id', 'origin', 'mase']], on=['item_id', 'origin'])
print(f"static MASE vs notebook 02: max abs difference {np.abs(chk['static'] - chk['mase']).max():.2e}")

mase = mase.merge(windows[['item_id', 'origin', 'has_break']], on=['item_id', 'origin'])
mase = mase.merge(hc[['item_id', 'origin', 'high_confidence']], on=['item_id', 'origin'], how='left')
mase['high_confidence'] = mase['high_confidence'].fillna(False).astype(bool)
mase['category'] = mase['item_id'].map(category)
mase.to_csv("../outputs/tables/adaptation_variants_item_origin_mase.csv", index=False)

def item_diffs(mask, variant):
    s = mase[mask]
    return s.assign(diff=s[variant] - s['static']).groupby(['item_id', 'category'], as_index=False)['diff'].mean()

rows = []
for group, mask in [('all detected breaks', mase['has_break']), ('high-confidence', mase['high_confidence'])]:
    block = []
    for v in VARIANTS:
        d = item_diffs(mask, v)
        block.append({'group': group, 'variant': v, **paired_test(d['diff'])})
    adj = holm_adjust([b['p_value'] for b in block])
    for b, a in zip(block, adj):
        b['p_holm'] = a
        b['descriptive_only'] = b['n'] < 20
        b['helps'] = bool(not b['descriptive_only'] and a < 0.05 and b['median_diff'] < 0 and abs(b['median_diff']) >= PRACTICAL)
    rows += block
tests = pd.DataFrame(rows)
tests.to_csv("../outputs/tables/adaptation_variants_tests.csv", index=False)
pd.set_option('display.width', 220)
cols = ['group', 'variant', 'n', 'median_diff', 'ci_low', 'ci_high', 'share_improved', 'rank_biserial', 'p_value', 'p_holm', 'helps']
print(tests[cols].round(4).to_string(index=False))

static MASE vs notebook 02: max abs difference 1.33e-15


              group     variant   n  median_diff  ci_low  ci_high  share_improved  rank_biserial  p_value  p_holm  helps
all detected breaks    truncate 488       0.0033  0.0015   0.0053          0.4324         0.2340   0.0000  0.0000  False
all detected breaks       decay 488       0.0011  0.0004   0.0024          0.4406         0.1042   0.0460  0.1381  False
all detected breaks level_shift 488       0.0001 -0.0012   0.0006          0.4959        -0.0268   0.6084  0.6084  False
all detected breaks       blend 488      -0.0010 -0.0018  -0.0001          0.5533        -0.0835   0.1099  0.2197  False
    high-confidence    truncate 254       0.0033  0.0008   0.0074          0.4252         0.2070   0.0042  0.0169  False
    high-confidence       decay 254       0.0010 -0.0008   0.0026          0.4528         0.0480   0.5076  1.0000  False
    high-confidence level_shift 254       0.0007 -0.0008   0.0019          0.4685         0.0475   0.5115  1.0000  False
    high-confidence       blend 

In [6]:
# Descriptive: by category (not part of the pre-registered decision rule)
rows = []
for group, mask in [('all detected breaks', mase['has_break']), ('high-confidence', mase['high_confidence'])]:
    for v in VARIANTS:
        d = item_diffs(mask, v)
        for cat, g in d.groupby('category'):
            rows.append({'group': group, 'variant': v, 'category': cat, **paired_test(g['diff'])})
bycat = pd.DataFrame(rows)
bycat.to_csv("../outputs/tables/adaptation_variants_by_category.csv", index=False)
print(bycat[['group', 'variant', 'category', 'n', 'median_diff', 'ci_low', 'ci_high', 'p_value']].round(4).to_string(index=False))
print("\nH3 (no variant helps on all detected breaks):",
      "SUPPORTED" if not tests[tests['group'] == 'all detected breaks']['helps'].any() else "NOT SUPPORTED")

              group     variant  category   n  median_diff  ci_low  ci_high  p_value
all detected breaks    truncate     FOODS 234       0.0054  0.0017   0.0087   0.0001
all detected breaks    truncate   HOBBIES  67       0.0036  0.0008   0.0082   0.0059
all detected breaks    truncate HOUSEHOLD 187       0.0011 -0.0024   0.0046   0.1562
all detected breaks       decay     FOODS 234       0.0007 -0.0005   0.0035   0.3325
all detected breaks       decay   HOBBIES  67       0.0007 -0.0011   0.0033   0.4687
all detected breaks       decay HOUSEHOLD 187       0.0019  0.0005   0.0029   0.0611
all detected breaks level_shift     FOODS 234      -0.0014 -0.0027   0.0003   0.0954
all detected breaks level_shift   HOBBIES  67      -0.0011 -0.0028   0.0006   0.1960
all detected breaks level_shift HOUSEHOLD 187       0.0014  0.0002   0.0022   0.0400
all detected breaks       blend     FOODS 234      -0.0014 -0.0029   0.0005   0.4315
all detected breaks       blend   HOBBIES  67       0.0005 -0.000